# Phase 1 - Data Audit

Notebook này kiểm tra chất lượng dữ liệu ban đầu cho project **E-commerce Sales & Customer Behavior Analytics**.

Mục tiêu:

- Hiểu cấu trúc raw dataset.
- Kiểm tra shape, schema, data types.
- Kiểm tra missing values và duplicate records.
- Kiểm tra numeric metrics như `Sales`, `Quantity`, `Discount`, `Profit`, `Shipping_Cost`.
- Kiểm tra categorical values như `Product_Category`, `Payment_method`, `Device_Type`.
- Export audit tables để dùng trong report và portfolio.

## 1. Import Libraries

In [1]:
from pathlib import Path
from typing import Iterable

import pandas as pd

## 2. Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "E-commerce Dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "data_audit_tables"

REQUIRED_COLUMNS = [
    "Order_Date",
    "Time",
    "Aging",
    "Customer_Id",
    "Gender",
    "Device_Type",
    "Customer_Login_type",
    "Product_Category",
    "Product",
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping_Cost",
    "Order_Priority",
    "Payment_method",
]

NUMERIC_COLUMNS = [
    "Aging",
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping_Cost",
]

CATEGORICAL_COLUMNS = [
    "Gender",
    "Device_Type",
    "Customer_Login_type",
    "Product_Category",
    "Order_Priority",
    "Payment_method",
]

## 3. Helper Functions

In [ ]:
def validate_columns(df: pd.DataFrame, required_columns: Iterable[str]) -> None:
    """Validate that the raw dataset contains all expected columns."""
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")


def load_raw_data(file_path: Path) -> pd.DataFrame:
    """Load raw CSV data and perform minimal schema validation."""
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset not found: {file_path}")

    df = pd.read_csv(file_path)
    validate_columns(df, REQUIRED_COLUMNS)
    return df


def prepare_audit_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Convert audit-critical fields to the right data types without dropping rows."""
    audit_df = df.copy()
    audit_df["Order_Date"] = pd.to_datetime(audit_df["Order_Date"], errors="coerce")

    for col in NUMERIC_COLUMNS:
        audit_df[col] = pd.to_numeric(audit_df[col], errors="coerce")

    return audit_df

## 4. Load Dataset

In [4]:
try:
    raw_df = load_raw_data(DATA_PATH)
    audit_df = prepare_audit_frame(raw_df)
    print(f"Dataset loaded successfully: {audit_df.shape[0]:,} rows x {audit_df.shape[1]} columns")
except Exception as error:
    print(f"Failed to load dataset: {error}")
    raise

Dataset loaded successfully: 51,290 rows x 16 columns


In [16]:
audit_df.head(5)

,Order_Date,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Product_Category,Product,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Payment_method
0,2018-01-02,10:56:33,8.0,37077,Female,Web,Member,Auto & Accessories,Car Media Players,140.0,1.0,0.3,46.0,4.6,Medium,credit_card
1,2018-07-24,20:41:37,2.0,59173,Female,Web,Member,Auto & Accessories,Car Speakers,211.0,1.0,0.3,112.0,11.2,Medium,credit_card
2,2018-11-08,8:38:49,8.0,41066,Female,Web,Member,Auto & Accessories,Car Body Covers,117.0,5.0,0.1,31.2,3.1,Critical,credit_card
3,2018-04-18,19:28:06,7.0,50741,Female,Web,Member,Auto & Accessories,Car & Bike Care,118.0,1.0,0.3,26.2,2.6,High,credit_card
4,2018-08-13,21:18:39,9.0,53639,Female,Web,Member,Auto & Accessories,Tyre,250.0,1.0,0.3,160.0,16.0,Critical,credit_card


## 5. Dataset Shape and Schema

In [6]:
shape_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "duplicate_rows", "unique_customers", "unique_products"],
        "value": [
            len(audit_df),
            audit_df.shape[1],
            audit_df.duplicated().sum(),
            audit_df["Customer_Id"].nunique(),
            audit_df["Product"].nunique(),
        ],
    }
)

shape_summary

,metric,value
0,rows,51290
1,columns,16
2,duplicate_rows,0
3,unique_customers,38997
4,unique_products,42


In [7]:
schema_summary = pd.DataFrame(
    {
        "column_name": audit_df.columns,
        "data_type": audit_df.dtypes.astype(str).values,
        "non_null_count": audit_df.notna().sum().values,
        "unique_count": audit_df.nunique(dropna=True).values,
    }
)

schema_summary

,column_name,data_type,non_null_count,unique_count
0,Order_Date,datetime64[us],51290,356
1,Time,str,51290,35275
2,Aging,float64,51289,11
3,Customer_Id,int64,51290,38997
4,Gender,str,51290,2
5,Device_Type,str,51290,2
6,Customer_Login_type,str,51290,4
7,Product_Category,str,51290,4
8,Product,str,51290,42
9,Sales,float64,51289,39


## 6. Date Coverage

In [ ]:
date_summary = pd.DataFrame(
    {
        "metric": ["min_order_date", "max_order_date", "invalid_order_dates"],
        "value": [
            audit_df["Order_Date"].min(),
            audit_df["Order_Date"].max(),
            audit_df["Order_Date"].isna().sum(),
        ],
    }
)

date_summary

,metric,value
0,min_order_date,2018-01-01 00:00:00
1,max_order_date,2018-12-30 00:00:00
2,invalid_order_dates,0


## 7. Missing Values

In [9]:
missing_values = (
    pd.DataFrame(
        {
            "column_name": audit_df.columns,
            "missing_count": audit_df.isna().sum().values,
            "missing_percent": (audit_df.isna().sum().values / len(audit_df) * 100).round(4),
        }
    )
    .query("missing_count > 0")
    .sort_values(["missing_count", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

missing_values

,column_name,missing_count,missing_percent
0,Order_Priority,2,0.0039
1,Quantity,2,0.0039
2,Aging,1,0.0019
3,Discount,1,0.0019
4,Sales,1,0.0019
5,Shipping_Cost,1,0.0019


In [10]:
missing_examples = audit_df.loc[audit_df.isna().any(axis=1)].copy()
missing_examples.insert(0, "csv_line_number", missing_examples.index + 2)
missing_examples["missing_columns"] = missing_examples.apply(
    lambda row: ", ".join(row.index[row.isna()].tolist()),
    axis=1,
)

missing_examples

,csv_line_number,Order_Date,Time,Aging,Customer_Id,Gender,Device_Type,Customer_Login_type,Product_Category,Product,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Payment_method,missing_columns
27,29,2018-05-02,11:45:38,NaN,26058,Female,Web,Member,Auto & Accessories,Car Media Players,140.0,1.0,0.3,55.8,5.6,High,credit_card,Aging
95,97,2018-04-22,11:32:22,5.0,52267,Male,Web,Member,Auto & Accessories,Bike Tyres,72.0,NaN,0.1,36.0,3.6,Critical,credit_card,Quantity
211,213,2018-08-05,17:27:54,6.0,47137,Male,Web,Member,Auto & Accessories,Tyre,250.0,5.0,NaN,132.5,13.3,Medium,credit_card,Discount
321,323,2018-06-05,11:04:11,3.0,41850,Male,Web,Member,Auto & Accessories,Car Mat,54.0,NaN,0.2,54.0,5.4,Critical,credit_card,Quantity
535,537,2018-04-16,16:20:02,3.0,13777,Male,Web,Member,Auto & Accessories,Tyre,250.0,4.0,0.2,150.0,NaN,Critical,credit_card,Shipping_Cost
625,627,2018-10-15,20:16:34,2.0,26367,Male,Web,Member,Auto & Accessories,Tyre,250.0,4.0,0.3,140.0,14.0,NaN,debit_card,Order_Priority
791,793,2018-07-03,23:40:16,4.0,36902,Female,Web,Member,Auto & Accessories,Car Pillow & Neck Rest,231.0,1.0,0.1,148.7,14.9,NaN,money_order,Order_Priority
793,795,2018-05-16,21:30:59,6.0,16381,Male,Web,Member,Auto & Accessories,Car Speakers,NaN,1.0,0.1,124.7,12.5,Critical,credit_card,Sales


## 8. Numeric Summary

In [11]:
numeric_summary = audit_df[NUMERIC_COLUMNS].describe().T.reset_index()
numeric_summary = numeric_summary.rename(columns={"index": "column_name"})
numeric_summary["missing_count"] = audit_df[NUMERIC_COLUMNS].isna().sum().values
numeric_summary["sum"] = audit_df[NUMERIC_COLUMNS].sum(numeric_only=True).values

numeric_summary = numeric_summary[
    ["column_name", "count", "missing_count", "mean", "std", "min", "25%", "50%", "75%", "max", "sum"]
].round(4)

numeric_summary

,column_name,count,missing_count,mean,std,min,25%,50%,75%,max,sum
0,Aging,51289.0,1,5.2550,2.9599,1.0,3.0,5.0,8.0,10.5,269525.5
1,Sales,51289.0,1,152.3409,66.4954,33.0,85.0,133.0,218.0,250.0,7813411.0
2,Quantity,51288.0,2,2.5030,1.5119,1.0,1.0,2.0,4.0,5.0,128373.0
3,Discount,51289.0,1,0.3038,0.1310,0.1,0.2,0.3,0.4,0.5,15582.7
4,Profit,51290.0,0,70.4072,48.7295,0.5,24.9,59.9,118.4,167.5,3611186.6
5,Shipping_Cost,51289.0,1,7.0416,4.8717,0.1,2.5,6.0,11.8,16.8,361154.4


## 9. Categorical Distribution

In [12]:
categorical_tables = []

for col in CATEGORICAL_COLUMNS:
    distribution = (
        audit_df[col]
        .fillna("<missing>")
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={col: "value", "count": "row_count"})
    )
    distribution.insert(0, "column_name", col)
    distribution["row_percent"] = (distribution["row_count"] / len(audit_df) * 100).round(4)
    categorical_tables.append(distribution)

categorical_distribution = pd.concat(categorical_tables, ignore_index=True)
categorical_distribution

,column_name,value,row_count,row_percent
0,Gender,Male,28138,54.8606
1,Gender,Female,23152,45.1394
2,Device_Type,Web,47632,92.8680
3,Device_Type,Mobile,3658,7.1320
4,Customer_Login_type,Member,49097,95.7243
5,Customer_Login_type,Guest,1993,3.8857
6,Customer_Login_type,First SignUp,173,0.3373
7,Customer_Login_type,New,27,0.0526
8,Product_Category,Fashion,25646,50.0019
9,Product_Category,Home & Furniture,15438,30.0994


## 10. Category KPI Summary

In [13]:
category_kpi_summary = (
    audit_df.groupby("Product_Category", dropna=False)
    .agg(
        orders=("Product", "count"),
        customers=("Customer_Id", "nunique"),
        products=("Product", "nunique"),
        revenue=("Sales", "sum"),
        quantity=("Quantity", "sum"),
        profit=("Profit", "sum"),
        shipping_cost=("Shipping_Cost", "sum"),
        avg_discount=("Discount", "mean"),
    )
    .reset_index()
)

category_kpi_summary["profit_margin"] = category_kpi_summary["profit"] / category_kpi_summary["revenue"]
category_kpi_summary["shipping_cost_ratio"] = category_kpi_summary["shipping_cost"] / category_kpi_summary["revenue"]
category_kpi_summary["revenue_share"] = category_kpi_summary["revenue"] / category_kpi_summary["revenue"].sum()

category_kpi_summary = category_kpi_summary.sort_values("revenue", ascending=False).round(4)
category_kpi_summary

,Product_Category,orders,customers,products,revenue,quantity,profit,shipping_cost,avg_discount,profit_margin,shipping_cost_ratio,revenue_share
2,Fashion,25646,22338,11,4345914.0,66639.0,2072623.9,207237.9,0.3560,0.4769,0.0477,0.5562
3,Home & Furniture,15438,12830,10,1975831.0,38190.0,880058.9,88054.5,0.2859,0.4454,0.0446,0.2529
0,Auto & Accessories,7505,7003,9,1096928.0,17593.0,484313.2,48426.2,0.2142,0.4415,0.0441,0.1404
1,Electronic,2701,2633,12,394738.0,5951.0,174190.6,17435.8,0.1597,0.4413,0.0442,0.0505


## 11. Monthly Revenue

In [14]:
monthly_revenue = (
    audit_df.assign(order_month=audit_df["Order_Date"].dt.to_period("M").astype(str))
    .groupby("order_month", dropna=False)
    .agg(
        orders=("Product", "count"),
        revenue=("Sales", "sum"),
        profit=("Profit", "sum"),
        customers=("Customer_Id", "nunique"),
    )
    .reset_index()
    .sort_values("order_month")
)

monthly_revenue["profit_margin"] = monthly_revenue["profit"] / monthly_revenue["revenue"]
monthly_revenue["revenue_mom_growth"] = monthly_revenue["revenue"].pct_change()
monthly_revenue = monthly_revenue.round(4)

monthly_revenue

,order_month,orders,revenue,profit,customers,profit_margin,revenue_mom_growth
0,2018-01,2519,379627.0,174573.6,2473,0.4599,NaN
1,2018-02,2206,332495.0,153288.2,2180,0.4610,-0.1242
2,2018-03,2899,435502.0,200936.8,2854,0.4614,0.3098
3,2018-04,3896,597312.0,277832.2,3798,0.4651,0.3715
4,2018-05,5417,824502.0,379386.3,5275,0.4601,0.3804
5,2018-06,4179,642555.0,298300.1,4085,0.4642,-0.2207
6,2018-07,5321,810205.0,374391.6,5204,0.4621,0.2609
7,2018-08,4373,664495.0,306904.0,4268,0.4619,-0.1798
8,2018-09,4851,738303.0,341558.1,4723,0.4626,0.1111
9,2018-10,4886,743387.0,342368.5,4740,0.4606,0.0069


## 12. Export Audit Tables

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_values.to_csv(OUTPUT_DIR / "missing_values.csv", index=False)
numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv", index=False)
categorical_distribution.to_csv(OUTPUT_DIR / "categorical_distribution.csv", index=False)
category_kpi_summary.to_csv(OUTPUT_DIR / "category_kpi_summary.csv", index=False)
monthly_revenue.to_csv(OUTPUT_DIR / "monthly_revenue.csv", index=False)
missing_examples.to_csv(OUTPUT_DIR / "missing_value_examples.csv", index=False)

print(f"Audit tables exported to: {OUTPUT_DIR}")

Audit tables exported to: c:\Users\Dell\Desktop\Portfolio\reports\data_audit_tables


## 13. Key Findings

- Dataset có 51,290 rows và 16 columns.
- Date range bao phủ gần trọn năm 2018.
- Không có duplicate rows.
- Missing values rất ít, nhưng nằm ở các cột business-critical như `Sales`, `Quantity`, `Discount`, `Shipping_Cost`.
- `Payment_method` có giá trị `not_defined`, nên cần chuẩn hóa trong Phase 2.
- `Fashion` là category lớn nhất theo revenue và profit.

Decision cho Phase 2: cần chuẩn hóa column names, xử lý missing values, chuẩn hóa categorical labels, và tạo thêm date/business features.